# PCB test-point placement — DreamerV3 cold-start training (A100, single board)

Trains the placement policy on the **TE-family board only** (`--boards central`: the canonical 20-trace connector board with per-seed jitter, no moat boards) with the full cold-start stack — expert demos, anchored behavior cloning (10% floor), potential reward shaping, single-layer reward, and the exact-geometry vector observation. Then scores the trained policy against the classical baselines. Everything — demos, replay, checkpoints, TensorBoard logs — is saved to your Google Drive as it runs.

**Setup:** `Runtime → Change runtime type → A100 GPU` (Colab Pro), then run all cells top to bottom. The Drive cell asks for authorization once.

Timeline: demo generation ~2–3 min (TE-family boards are fast; one-time, cached in Drive), then 60k training steps ≈ 2–4 h with checkpoints every 5k steps. Interrupt — or lose the runtime — anytime: re-running the notebook resumes exactly where it left off. On a free T4 instead, set `CONFIG = "colab"` and `NUM_TRACES = 8` in the settings cell.

In [ ]:
!nvidia-smi -L
import torch
assert torch.cuda.is_available(), "No GPU — set Runtime > Change runtime type > A100 GPU"
name = torch.cuda.get_device_name(0)
print("torch", torch.__version__, "|", name)
if "A100" not in name:
    print("NOTE: not an A100 — everything still runs, just slower; "
          "consider CONFIG='colab', NUM_TRACES=8 in the settings cell.")

In [ ]:
# All logs, demos, and checkpoints live in your Google Drive, so nothing is
# lost on disconnect and re-running this notebook later resumes training.
from google.colab import drive
drive.mount("/content/drive")
LOGROOT = "/content/drive/MyDrive/pcb-router-logs"
print("backing up to:", LOGROOT)

In [ ]:
# Get the code from GitHub main (pulls the latest on re-runs).
import pathlib
if not pathlib.Path("/content/pcb-router").exists():
    !git clone -q https://github.com/pauljiang03/pcb-router /content/pcb-router
%cd /content/pcb-router
!git pull --ff-only

In [ ]:
# Dependencies (torch/numpy/tensorboard/matplotlib ship with Colab) and a
# quick sanity run of the cold-start tests (~10 s).
%pip -q install gymnasium "ruamel.yaml" openpyxl
!python -m pytest tests/test_coldstart.py -q

In [ ]:
NUM_TRACES = 20        # canonical board size (T4 fallback: 8)
CONFIG = "colab_a100"  # configs.yaml section (T4 fallback: "colab")
BOARDS = "central"     # TE-family board only; "mixed" adds moat challenge boards
STEPS = 60000          # enough for the single-board task
# 4 env workers overlap CPU routing with GPU training (A100 VMs have 12 vCPUs).
# Set to "" if the worker processes misbehave.
ENV_FLAGS = "--envs 4 --parallel"
# Fresh run dir: the obs format changed (vector key), so never reuse logdirs
# from runs before commit c10d084.
RUN_DIR = f"{LOGROOT}/central"
print(NUM_TRACES, "traces |", CONFIG, "|", BOARDS, "|", STEPS, "steps |", RUN_DIR)

In [ ]:
# Live training curves. Key scalars:
#   log_routable  -- the live planarity metric: +10 = all traces routed on ONE
#                    layer, -5 per planar failure; want the average -> +10
#   bc_loss       -- imitation fit; should fall toward ~1 and never vanish
#                    (bc_floor keeps a permanent 10% anchor)
#   imag_reward_mean -- exploitation alarm: real per-step reward can't exceed
#                    ~+2; if this balloons, the model is hallucinating again
#   eval_return   -- compare against the demo running-mean printed during
#                    demo generation (that number IS the baseline anchor)
%load_ext tensorboard
import tensorboard.notebook as tbnb
tbnb.start("--logdir " + LOGROOT)

## Train — single board, cold start (demos + anchored BC + shaping)

First run generates 200 expert episodes into `demo_eps/` in your Drive (~2–3 min for TE-family boards, 8 parallel workers). **Note the running-mean return it prints — that's your baseline anchor for eval.** Then it trains 60k steps. Interrupt anytime; re-run this cell to resume — finished demos are never regenerated.

In [ ]:
!python train.py --configs defaults {CONFIG} --logdir "{RUN_DIR}" \
    --num_traces {NUM_TRACES} --device cuda:0 {ENV_FLAGS} \
    --boards {BOARDS} --steps {STEPS}

In [ ]:
# Score the trained policy against the classical baselines on the SAME boards
# (the summary table at the bottom is the headline result: compare the
# Dreamer row to Smart on failures / max / spread). Default = the fixed TE
# board (what this run trains for); --board_seed 1000000 scores held-out
# TE-family variants; --fast for a quick low-budget pass.
!python eval.py --checkpoint "{RUN_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 3 --num_traces {NUM_TRACES} --device cuda:0 --no-plot

In [ ]:
# Render the routed boards to PNGs (~2-3 min: same quality router as the
# table above, one episode per method), save them to Drive, and display the
# headline comparison inline: Smart (classical) vs Dreamer (learned).
!python eval.py --checkpoint "{RUN_DIR}/latest.pt" --configs defaults {CONFIG} \
    --episodes 1 --num_traces {NUM_TRACES} --device cuda:0

import pathlib, shutil
from IPython.display import Image, display
figs = pathlib.Path(RUN_DIR) / "figs"
figs.mkdir(exist_ok=True)
for p in sorted(pathlib.Path("eval_results").glob("*_1.png")):
    shutil.copy(p, figs / p.name)
print("figures saved to", figs)
for name in ("smart_1.png", "dreamer_1.png"):
    p = pathlib.Path("eval_results") / name
    if p.exists():
        print("\n===", name, "===")
        display(Image(str(p)))

## Reading the results

- **`log_routable`** — the planarity metric. +10 = all 20 traces routed on the single copper layer; each planar failure costs −5. The demos sit at +10; the policy's average should climb there and stay.
- **`bc_loss`** — imitation fit. Starts ≈5 (uniform over 200 candidates); with the vector observation it should fall steadily toward ~1. It never disappears — `bc_floor` keeps a permanent 10% anchor precisely so the policy can't drift into states where the world model hallucinates.
- **`imag_reward_mean`** — the exploitation alarm. Real per-step reward tops out around +2; if this balloons to +10 or more, the reward head is hallucinating again and the run is suspect.
- **`eval_return` vs the demo anchor** — the demo generator's running-mean return is the baseline. Reaching it = the cold start worked; exceeding it = RL is finding shorter, better-matched placements than the heuristic (watch `log_length_max` and `log_spread` shrink toward 0 — that's the real objective on this already-routable board).
- **`eval.py` summary table** — the scoreboard vs. classical baselines. Beating Smart on `max=` / `spread=` on the TE board is the win condition for the single-board model. Watch that failures never rise while return improves.

Everything is already backed up in your Drive — stopping the runtime loses nothing.

In [ ]:
# Optional: download a local copy of the checkpoint + TensorBoard events.
# (Your Drive already has everything.)
import pathlib, shutil
out = pathlib.Path("/content/results")
shutil.rmtree(out, ignore_errors=True)
out.mkdir(parents=True)
d = pathlib.Path(RUN_DIR)
for f in list(d.glob("events*")) + [d / "latest.pt"]:
    if f.exists():
        shutil.copy(f, out / f.name)
shutil.make_archive("/content/pcb_router_results", "zip", out)
from google.colab import files
files.download("/content/pcb_router_results.zip")